In [1]:
import pandas as pd
from difflib import get_close_matches
import os
# --- Load datasets ---
growth = pd.read_csv(os.path.join("Data","growth.csv"))
bace = pd.read_excel(os.path.join("Data","BACE_data.xls"))  # uses first sheet by default

# normalize column names (just trim)
growth.columns = [c.strip() for c in growth.columns]
bace.columns = [c.strip() for c in bace.columns]

# --- Map same-concept variables to avoid duplicating them ---
# BACE shortname -> growth.csv column
overlap_map = {
    "ABSLATIT": "abslat",
    "CATH00": "pcatholic",
    "MUSLIM00": "pmuslim",
    "PROT00": "pprotest",
    "LANDAREA": "area",
    "LANDLOCK": "landlock",
    "EUROPE": "europe",
    "SAFRICA": "africa",
    "COLONY": "excolony",
    "TROPICAR": "tropicar",
    # If you consider these equivalent, keep commented; else add to import set:
    # "LHCPC": "oilres",
}

# --- BACE vars we want to import if they exist and aren't overlapped ---
bace_candidates = {
    "AIRDIST","AVELF","BRIT","BUDDHA","CIV72","CONFUC","DENS60","DENS65C","DENS65I",
    "DPOP6090","EAST","ECORG","ENGFRAC","FERTLDC1","GDE1","GDPCH60L","GEEREC1",
    "GGCFD3","GOVNOM1","GOVSH61","GVR61","H60","HERF00","HINDU00","IPRICE1","LAAM",
    "LIFE060","LT100CR","MALFAL66","MINING","NEWSTATE","OIL","OPENDEC1","ORTH00",
    "OTHFRAC","P60","PI6090","SQPI6090","PRIGHTS","POP1560","POP60","POP6560",
    "PRIEXP70","RERD","REVCOUP","SCOUT","SIZE60","SOCIALIST","SPAIN","TOT1DEC1",
    "TOTIND","TROPPOP","WARTIME","WARTORN","YRSOPEN","ZTROPICS"
}

# Remove anything conceptually covered by growth via overlap_map
bace_candidates -= set(overlap_map.keys())

# Keep only those BACE vars that are actually present
bace_existing = set(bace.columns)
missing_from_growth = sorted(list(bace_candidates.intersection(bace_existing)))

# --- Build slice & merge on 'code' ---
if "code" not in growth.columns or "code" not in bace.columns:
    raise ValueError("Both datasets must have a 'code' column for the merge you requested.")

bace_slice = bace[["code"] + missing_from_growth].copy()
merged = growth.merge(bace_slice, how="left", on="code")

# --- Save & report ---
merged.to_csv(os.path.join("Data","growth_plus_bace_missing.csv"), index=False)
print("Added columns from BACE:", missing_from_growth)
print("Saved -> Data/growth_plus_bace_missing.csv")

Added columns from BACE: ['AIRDIST', 'AVELF', 'BRIT', 'BUDDHA', 'CIV72', 'CONFUC', 'DENS60', 'DENS65C', 'DENS65I', 'DPOP6090', 'EAST', 'ECORG', 'ENGFRAC', 'FERTLDC1', 'GDE1', 'GDPCH60L', 'GEEREC1', 'GGCFD3', 'GOVNOM1', 'GOVSH61', 'GVR61', 'H60', 'HERF00', 'HINDU00', 'IPRICE1', 'LAAM', 'LIFE060', 'LT100CR', 'MALFAL66', 'MINING', 'NEWSTATE', 'OIL', 'OPENDEC1', 'ORTH00', 'OTHFRAC', 'P60', 'PI6090', 'POP1560', 'POP60', 'POP6560', 'PRIEXP70', 'PRIGHTS', 'RERD', 'REVCOUP', 'SCOUT', 'SIZE60', 'SOCIALIST', 'SPAIN', 'SQPI6090', 'TOT1DEC1', 'TOTIND', 'TROPPOP', 'WARTIME', 'WARTORN', 'YRSOPEN', 'ZTROPICS']
Saved -> Data/growth_plus_bace_missing.csv
